# RAG-Sequence:
Ce jupyter reprend le code du rag naif mais change la géénratio nde la réponse pour que ça soit un RAG-Sequence

# 1) Imports

In [1]:
import os
import json
import ast
import re
import gc

from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import evaluate
from datasets import load_dataset
from tqdm import tqdm

import torch
import torch.nn.functional as F

import faiss
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)

import warnings
warnings.filterwarnings(
    "ignore",
    message=r".*_check_is_size will be removed in a future PyTorch release.*",
    category=FutureWarning,
)

/home/onyxia/work/RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2) Config (Qwen 14B uniquement + Retriever Qwen 0.6B)

In [2]:
# -------------------------
# Config
# -------------------------
# LLM (generator) - Qwen 14B ONLY
LLM_NAME, LLM_TAG = "JungZoona/T3Q-qwen2.5-14b-v1.0-e3", "14B"

# Dense retriever
RETRIEVER_NAME = "Qwen/Qwen3-Embedding-0.6B"

DATA_TRAIN = "data/train.csv"
DATA_VALID = "data/validation.csv"
SEED = 42

# Retrieval / indexing
TOP_K = 5
CHUNK_WORDS = 200
CHUNK_OVERLAP = 50
RET_MAX_TOKENS = 512
RET_BATCH_SIZE = 64
QUERY_BATCH_SIZE = 64

# Generation
MAX_INPUT_TOKENS = 768
MAX_NEW_TOKENS = 32
GEN_BATCH_SIZE = 1

DEVICE = "cuda"

# Cache
CACHE_DIR = "results/rag_cache"
RETR_TAG = RETRIEVER_NAME.split("/")[-1]
INDEX_PATH = os.path.join(
    CACHE_DIR,
    f"faiss_ip_{RETR_TAG}_dim1024_val_cw{CHUNK_WORDS}_ov{CHUNK_OVERLAP}.index"
)
CHUNKS_PATH = os.path.join(
    CACHE_DIR,
    f"chunks_{RETR_TAG}_val_cw{CHUNK_WORDS}_ov{CHUNK_OVERLAP}.jsonl"
)

# Outputs
METRICS_CSV = "results/metrics_table_qwen14B_rag_sequence_validation.xlsx"
PRED200_CSV = "results/manual_eval_qwen14B_rag_sequence_validation.xlsx"

# 3) Parsing dataset + prompt + nettoyage génération

In [3]:
def parse_answers_field(x):
    s = str(x).strip()
    s2 = re.sub(r"array\(\s*(\[[\s\S]*?\])\s*,\s*dtype=[^)]+\)", r"\1", s)
    ans = ast.literal_eval(s2)
    ans["text"] = list(ans["text"])
    ans["answer_start"] = [int(v) for v in ans["answer_start"]]
    return ans


def normalize_dataset(example):
    example["answers"] = parse_answers_field(example["answers"])
    example["id"] = str(example["id"])
    return example


def build_prompt(question, tokenizer, context=None):
    question = str(question).strip()

    system_msg = (
        "You are a helpful assistant. "
        "Answer the question with a short phrase. "
        "Do not add explanations. "
        "If the context does not contain the answer, say: I don't know."
    )

    if context is not None and len(context.strip()) > 0:
        user_msg = f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    else:
        user_msg = f"Question: {question}\nAnswer:"

    if hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    return f"{system_msg}\n\n{user_msg}"


def clean_generation(text):
    t = (text or "").strip()
    t = t.split("\n")[0].strip()
    for pref in ["Answer:", "A:", "assistant:", "Assistant:"]:
        if t.lower().startswith(pref.lower()):
            t = t[len(pref):].strip()
    return t.strip()

# 4) Chunking + Embeddings + FAISS

In [4]:
def chunk_text_words(text, chunk_words, overlap):
    text = (text or "").strip()
    if not text:
        return []
    words = text.split()
    if len(words) <= chunk_words:
        return [text]
    chunks = []
    step = max(1, chunk_words - overlap)
    for start in range(0, len(words), step):
        end = start + chunk_words
        chunk = " ".join(words[start:end]).strip()
        if chunk:
            chunks.append(chunk)
        if end >= len(words):
            break
    return chunks


def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
    summed = (last_hidden * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-9)
    return summed / denom


@torch.inference_mode()
def embed_texts(
    texts,
    tokenizer,
    model,
    max_tokens=512,
    batch_size=32,
    prefix=None,
    device="cuda",
):
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        if prefix is not None:
            batch = [prefix + t for t in batch]
        enc = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_tokens,
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(**enc)
        pooled = mean_pool(out.last_hidden_state, enc["attention_mask"])
        pooled = F.normalize(pooled, p=2, dim=1)
        vecs.append(pooled.detach().cpu().numpy().astype(np.float32))
        del enc, out, pooled
    return np.vstack(vecs) if vecs else np.zeros((0, 1024), dtype=np.float32)


def save_chunks_jsonl(path, chunks):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for t in chunks:
            f.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")


def load_chunks_jsonl(path):
    chunks = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            chunks.append(obj["text"])
    return chunks


def build_or_load_faiss_index(ds_corpus, retr_tok, retr_model, dim=1024):
    os.makedirs(CACHE_DIR, exist_ok=True)

    if os.path.exists(INDEX_PATH) and os.path.exists(CHUNKS_PATH):
        index = faiss.read_index(INDEX_PATH)
        chunks = load_chunks_jsonl(CHUNKS_PATH)
        return index, chunks

    raw_contexts = ds_corpus["context"]
    seen = set()
    corpus_chunks = []

    for ctx in raw_contexts:
        ctx = (ctx or "").strip()
        if not ctx:
            continue
        if ctx in seen:
            continue
        seen.add(ctx)
        corpus_chunks.extend(chunk_text_words(ctx, CHUNK_WORDS, CHUNK_OVERLAP))

    index = faiss.IndexFlatIP(dim)

    for i in tqdm(range(0, len(corpus_chunks), RET_BATCH_SIZE), desc="Indexing corpus (embeddings + FAISS)"):
        batch = corpus_chunks[i : i + RET_BATCH_SIZE]
        emb = embed_texts(
            batch,
            tokenizer=retr_tok,
            model=retr_model,
            max_tokens=RET_MAX_TOKENS,
            batch_size=min(RET_BATCH_SIZE, 64),
            prefix="passage: ",
            device=DEVICE,
        )
        if emb.shape[0] > 0:
            index.add(emb)

    faiss.write_index(index, INDEX_PATH)
    save_chunks_jsonl(CHUNKS_PATH, corpus_chunks)
    return index, corpus_chunks

# 5) Retrieval top-k (retourne concat + topk indices/scores/chunks)

In [5]:
def retrieve_topk_contexts(
    questions,
    retr_tok,
    retr_model,
    index,
    chunks,
    top_k=5,
):
    retrieved_contexts = []
    topk_indices_all = []
    topk_scores_all = []
    topk_chunks_all = []

    for i in tqdm(range(0, len(questions), QUERY_BATCH_SIZE), desc="Retrieving top-k contexts"):
        batch_q = questions[i : i + QUERY_BATCH_SIZE]
        q_emb = embed_texts(
            batch_q,
            tokenizer=retr_tok,
            model=retr_model,
            max_tokens=RET_MAX_TOKENS,
            batch_size=min(QUERY_BATCH_SIZE, 64),
            prefix="query: ",
            device=DEVICE,
        )

        D, I = index.search(q_emb, top_k)

        for scores_row, idx_row in zip(D, I):
            idx_list = []
            score_list = []
            text_list = []
            for s, idx in zip(scores_row, idx_row):
                idx = int(idx)
                if 0 <= idx < len(chunks):
                    idx_list.append(idx)
                    score_list.append(float(s))
                    text_list.append(chunks[idx])

            retrieved_contexts.append("\n\n".join(text_list))  # concat (pour hit rate, logs, etc.)
            topk_indices_all.append(idx_list)
            topk_scores_all.append(score_list)
            topk_chunks_all.append(text_list)

    return retrieved_contexts, topk_indices_all, topk_scores_all, topk_chunks_all

# 6) Charger le LLM Qwen 14B

In [6]:
def load_llm_qwen_14b():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    tok = AutoTokenizer.from_pretrained(LLM_NAME, use_fast=True)
    tok.padding_side = "left"
    tok.truncation_side = "left"
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

    llm = AutoModelForCausalLM.from_pretrained(
        LLM_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    llm.eval()
    llm.config.use_cache = False
    return tok, llm

# 7) Main: seed + load dataset

In [7]:
set_seed(SEED)
os.makedirs("results", exist_ok=True)

ds = load_dataset("csv", data_files={"train": DATA_TRAIN, "validation": DATA_VALID})
ds = ds.map(normalize_dataset)

val_split = ds["validation"]

# 8) Load retriever + build/load index + retrieve top-k

In [8]:
retr_tok = AutoTokenizer.from_pretrained(RETRIEVER_NAME, use_fast=True)
retr_model = AutoModel.from_pretrained(RETRIEVER_NAME, torch_dtype=torch.float16)
retr_model.eval().to("cuda")

index, corpus_chunks = build_or_load_faiss_index(val_split, retr_tok, retr_model, dim=1024)

val_questions = list(val_split["question"])
retrieved_ctxs, topk_idx, topk_scores, topk_texts = retrieve_topk_contexts(
    val_questions, retr_tok, retr_model, index, corpus_chunks, top_k=TOP_K
)

`torch_dtype` is deprecated! Use `dtype` instead!
Retrieving top-k contexts: 100%|██████████| 16/16 [00:04<00:00,  3.28it/s]


# 9) Hit rate + libérer le retriever (VRAM)

In [9]:
def normalize_text(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip().lower())

hits = 0
total = len(val_split)
for i in range(total):
    gold = val_split[i]["answers"]["text"][0] if val_split[i]["answers"]["text"] else ""
    if gold and normalize_text(gold) in normalize_text(retrieved_ctxs[i]):
        hits += 1

retrieval_hit_rate = 100.0 * hits / max(1, total)
print(f"Retrieval hit rate (gold in retrieved context): {retrieval_hit_rate:.2f}%")

# Free retriever
del retr_model
del retr_tok
torch.cuda.empty_cache()
gc.collect()

Retrieval hit rate (gold in retrieved context): 89.90%


978

# 10) Load LLM 14B

In [10]:
llm_tok, llm = load_llm_qwen_14b()

Loading checkpoint shards: 100%|██████████| 6/6 [01:13<00:00, 12.33s/it]


# 11) RAG-Sequence (générer 1 réponse par chunk + score “retriever + logprob”)

In [ ]:
squad_metric = evaluate.load("squad")
predictions = []

ids = list(val_split["id"])
gold_answers = [ex["answers"]["text"][0] if ex["answers"]["text"] else "" for ex in val_split]


selected_ctx_used = []
selected_k = []
selected_faiss_score = []
selected_combined_score = []

def softmax_np(x):
    x = np.array(x, dtype=np.float64)
    x = x - np.max(x)
    e = np.exp(x)
    s = e.sum()
    return (e / s) if s > 0 else np.ones_like(x) / max(1, len(x))

def seq_logprob_from_generate(gen_scores, gen_token_ids):
    # log p(y|x,chunk) = somme log-softmax au token généré
    if not gen_scores or not gen_token_ids:
        return 0.0
    total_logp = 0.0
    for step_logits, tok_id in zip(gen_scores, gen_token_ids):
        logp = torch.log_softmax(step_logits[0], dim=-1)[tok_id].item()
        total_logp += logp
    return float(total_logp)

for i in tqdm(range(len(val_split)), desc="Generating (RAG-Sequence, Qwen 14B)"):
    ex_id = str(ids[i])
    question = val_questions[i]

    chunks_i = topk_texts[i]
    scores_i = topk_scores[i]

    if not chunks_i:
        pred = "I don't know."
        predictions.append({"id": ex_id, "prediction_text": pred})
        selected_ctx_used.append("")
        selected_k.append(-1)
        selected_faiss_score.append(None)
        selected_combined_score.append(None)
        continue

    w_i = softmax_np(scores_i)

    best_pred = "I don't know."
    best_k = 0
    best_ctx = chunks_i[0]
    best_faiss = float(scores_i[0])
    best_score = -1e18

    # 1 génération par chunk
    for k, (chunk, faiss_s, w) in enumerate(zip(chunks_i, scores_i, w_i)):
        prompt = build_prompt(question, llm_tok, context=chunk)

        inputs = llm_tok(
            [prompt],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS,
        )
        inputs = {kk: vv.to(llm.device) for kk, vv in inputs.items()}

        with torch.inference_mode():
            gen_out = llm.generate(
                **inputs,
                use_cache=False,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                num_beams=1,
                eos_token_id=llm_tok.eos_token_id,
                pad_token_id=llm_tok.pad_token_id,
                return_dict_in_generate=True,
                output_scores=True,
            )

        prompt_len = inputs["input_ids"].shape[1]
        seq = gen_out.sequences
        gen_ids = seq[:, prompt_len:]
        gen_token_ids = gen_ids[0].tolist() if gen_ids.numel() > 0 else []

        gen_text = llm_tok.batch_decode(gen_ids, skip_special_tokens=True)[0] if gen_ids.numel() > 0 else ""
        cand_pred = clean_generation(gen_text)
        if cand_pred == "":
            cand_pred = "I don't know."

        logp_y = seq_logprob_from_generate(gen_out.scores, gen_token_ids)
        comb = logp_y + float(np.log(w + 1e-12))

        if comb > best_score:
            best_score = comb
            best_pred = cand_pred
            best_k = k
            best_ctx = chunk
            best_faiss = float(faiss_s)

        del inputs, gen_out, seq, gen_ids
        torch.cuda.empty_cache()

    predictions.append({"id": ex_id, "prediction_text": best_pred})
    selected_ctx_used.append(best_ctx)
    selected_k.append(int(best_k))
    selected_faiss_score.append(best_faiss)
    selected_combined_score.append(float(best_score))

Generating (RAG-Sequence, Qwen 14B):   0%|          | 0/1000 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


# 12) Évaluation EM/F1

In [ ]:
references = [{"id": str(ex["id"]), "answers": ex["answers"]} for ex in val_split]
results = squad_metric.compute(predictions=predictions, references=references)

metrics_df = pd.DataFrame([{
    "split": "validation",
    "exact_match": results["exact_match"],
    "f1": results["f1"],
    "model": "Qwen 14B",
    "retriever": RETRIEVER_NAME,
    "top_k": TOP_K,
    "chunk_words": CHUNK_WORDS,
    "chunk_overlap": CHUNK_OVERLAP,
    "retrieval_hit_rate": retrieval_hit_rate
}])

print("\n=== RAG-Sequence (Qwen 14B) Validation Results ===")
print(metrics_df.to_string(index=False))

# 13) Sauvegarde (métriques + fichier 200 avec contexte utilisé + topk)

In [ ]:
metrics_df.to_excel(METRICS_CSV, index=False)

pred_by_id = {p["id"]: p["prediction_text"] for p in predictions}
pred200 = []

for i in range(min(200, len(val_split))):
    ex_id = str(ids[i])
    pred200.append({
        "id": ex_id,
        "question": val_questions[i],
        "gold_answer": gold_answers[i],
        "generated_answer": pred_by_id.get(ex_id, ""),
        "retrieved_ctx_used": selected_ctx_used[i],
    })

pd.DataFrame(pred200).to_excel(PRED200_CSV, index=False)

print("\nSaved:")
print(" -", METRICS_CSV)
print(" -", PRED200_CSV)